In [1]:
# ==============================================================================
# PARAMÈTRES EXIGÉS PAR LE PROJET (Deliverable 2)
# ==============================================================================
query = "What is a Neural Network?"
embedding_strategy = "sbert"                # Choix: "sbert", "word2vec", "tfidf"
similarity_strategy = "cosine"              # Choix: "cosine", "euclidean"
number_of_text_chunks_to_retrieve = 4
number_of_questions_to_generate = 2
generator_backend = "ollama"                
generator_model = "llama3.2:3b"
max_context_tokens = 2048                   # Limite de sécurité pour l'IA
input_dir = "/home/guest/shared/course_material/"
output_dir = "/home/guest/shared/course_material/output/"

print("✅ Paramètres du système RAG chargés.")

✅ Paramètres du système RAG chargés.


In [2]:
# ==============================================================================
# CELLULE 1 : INITIALISATION SPARK ET FONCTIONS DE DISTANCE (UDFS)
# ==============================================================================
import os
import pickle
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

from pyspark.sql import SparkSession
from pyspark.sql.functions import udf, col, desc, asc
from pyspark.sql.types import FloatType

# 1. Démarrage de la session Spark (Le "Driver")
spark = SparkSession.builder \
    .appName("RAG_Retrieval_Engine") \
    .master("local[*]") \
    .getOrCreate()

# Désactivation du parallélisme interne d'HuggingFace pour éviter les conflits avec Spark
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. Fonctions mathématiques distribuées (UDFs)
# L'utilisation d'UDFs permet d'exécuter le calcul sur tous les noeuds Spark
# en parallèle (O(N) distribué), évitant un .collect() prématuré de toute la base.

def make_spark_cosine_udf(query_vector):
    """Calcule la similarité Cosinus (Plus proche de 1 = Meilleur)."""
    def compute_cosine(doc_emb):
        if doc_emb is None: return 0.0
        q = np.array(query_vector, dtype=np.float32)
        d = np.array(doc_emb, dtype=np.float32)
        
        norm_q = np.linalg.norm(q)
        norm_d = np.linalg.norm(d)
        if norm_q == 0 or norm_d == 0: return 0.0
            
        return float(np.dot(q, d) / (norm_q * norm_d))
    return udf(compute_cosine, FloatType())

def make_spark_euclidean_udf(query_vector):
    """Calcule la distance Euclidienne (Plus proche de 0 = Meilleur)."""
    def compute_euclidean(doc_emb):
        if doc_emb is None: return 999.0 # Valeur arbitraire haute pour repousser les erreurs
        q = np.array(query_vector, dtype=np.float32)
        d = np.array(doc_emb, dtype=np.float32)
        
        # Formule de la distance euclidienne : racine de la somme des carrés des différences
        return float(np.linalg.norm(q - d))
    return udf(compute_euclidean, FloatType())

print("✅ Moteur Spark initialisé et UDFs mathématiques prêtes.")

/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/guest/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 19:30:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/20 19:30:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


✅ Moteur Spark initialisé et UDFs mathématiques prêtes.


In [3]:
# ==============================================================================
# CELLULE 2 : FONCTION DE RETRIEVAL (O(N) distribué)
# ==============================================================================

def retrieve_top_chunks(query: str, embedding_strategy: str, similarity_metric: str, k: int = 4):
    """
    Recherche les K meilleurs paragraphes selon la stratégie et la métrique choisies.
    """
    print(f"🔍 Recherche en cours... [Stratégie: {embedding_strategy} | Métrique: {similarity_metric}]")
    
    # 1. Chargement dynamique de la base de données Parquet appropriée
    parquet_path = f"/home/guest/shared/course_material/embeddings_{embedding_strategy}.parquet"
    
    try:
        df_corpus = spark.read.parquet(parquet_path)
    except Exception as e:
        print(f"❌ Erreur : Impossible de lire le fichier Parquet pour {embedding_strategy}.")
        return []

    # 2. Vectorisation de la question utilisateur (Doit matcher la stratégie d'entraînement)
    if embedding_strategy == "sbert":
        encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        q_vec = encoder.encode(query, normalize_embeddings=True)
        
    elif embedding_strategy == "word2vec":
        from pyspark.ml.feature import Word2VecModel, Tokenizer
        
        # 1. Spark MLlib a besoin d'un mini DataFrame pour transformer la question
        df_query = spark.createDataFrame([(query,)], ["chunk_text"])
        
        # 2. On découpe la question en mots (exactement comme dans la Tâche 1)
        tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
        wordsData = tokenizer.transform(df_query)
        
        # 3. On charge le modèle Word2Vec de SPARK (il comprend très bien les dossiers !)
        w2v_model = Word2VecModel.load("/home/guest/shared/course_material/w2v_model.model")
        
        # 4. On calcule le vecteur de la question
        w2vData = w2v_model.transform(wordsData)
        
        # 5. On extrait le vecteur final sous forme de tableau NumPy
        q_vec = w2vData.select("features").first()[0].toArray()
        
    elif embedding_strategy == "tfidf":
        from pyspark.ml.feature import IDFModel, Tokenizer, HashingTF
        
        # Spark MLlib a besoin d'un mini DataFrame pour transformer la question
        df_query = spark.createDataFrame([(query,)], ["chunk_text"])
        
        # On repasse la question dans les mêmes filtres que la Tâche 1
        tokenizer = Tokenizer(inputCol="chunk_text", outputCol="words")
        wordsData = tokenizer.transform(df_query)
        
        hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=20000)
        featurizedData = hashingTF.transform(wordsData)
        
        # On charge ton cerveau sauvegardé
        idfModel = IDFModel.load("/home/guest/shared/course_material/tfidf_model.model")
        rescaledData = idfModel.transform(featurizedData)
        
        # On extrait le vecteur final (on le convertit en array classique pour le calcul)
        q_vec = rescaledData.select("features").first()[0].toArray()
    
    else:
        raise ValueError(f"Stratégie inconnue : {embedding_strategy}")

    # 3. Application de la bonne métrique mathématique et tri
    if similarity_metric == "cosine":
        # Cosinus : On trie par ordre décroissant (le plus grand score gagne)
        metric_udf = make_spark_cosine_udf(q_vec)
        df_scored = df_corpus.withColumn("score", metric_udf(col("embedding")))
        df_top_k = df_scored.orderBy(desc("score")).limit(k)
        
    elif similarity_metric == "euclidean":
        # Euclidienne : On trie par ordre croissant (la plus petite distance gagne)
        metric_udf = make_spark_euclidean_udf(q_vec)
        df_scored = df_corpus.withColumn("score", metric_udf(col("embedding")))
        df_top_k = df_scored.orderBy(asc("score")).limit(k)
        
    else:
        raise ValueError(f"Métrique inconnue : {similarity_metric}")
    
    # 4. Rapatriement local des K résultats UNIQUEMENT (Conforme aux consignes)
    return df_top_k.collect()

# --- TEST DE LA FONCTION ---
ma_question = "What is a Neural Network?"

# On teste avec SBERT et Cosinus (Tu peux changer les paramètres pour tester Euclidienne)
resultats = retrieve_top_chunks(
    query=query, 
    embedding_strategy=embedding_strategy, 
    similarity_metric=similarity_strategy, 
    k=number_of_text_chunks_to_retrieve
)

for i, row in enumerate(resultats, 1):
    print(f"TOP {i} [Score: {row['score']:.4f}] - Doc: {row['source_pdf']} (p.{row['page_num']})")

🔍 Recherche en cours... [Stratégie: sbert | Métrique: cosine]


🎯 TOP 1 [Score: 0.6328] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
🎯 TOP 2 [Score: 0.5829] - Doc: A Neural Network Playground.pdf (p.1)
🎯 TOP 3 [Score: 0.5711] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
🎯 TOP 4 [Score: 0.5706] - Doc: Intro_DL_H515_2026.pdf (p.25)


In [6]:
"""
Question = "What is a Neural Network?"

score Sbert
TOP 1 [Score: 0.6328] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
TOP 2 [Score: 0.5829] - Doc: A Neural Network Playground.pdf (p.1)
TOP 3 [Score: 0.5711] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)
TOP 4 [Score: 0.5706] - Doc: Intro_DL_H515_2026.pdf (p.25)


score word2vec
TOP 1 [Score: 0.1938] - Doc: 1-intro.pdf (p.2)
TOP 2 [Score: 0.1741] - Doc: intro.pdf (p.3)
TOP 3 [Score: 0.1694] - Doc: 1-intro.pdf (p.19)
TOP 4 [Score: 0.1607] - Doc: 1-intro.pdf (p.30)


score tfidf
TOP 1 [Score: 0.1719] - Doc: Intro_DL_H515_2026.pdf (p.6)
TOP 2 [Score: 0.1466] - Doc: A Neural Network Playground.pdf (p.1)
TOP 3 [Score: 0.1329] - Doc: 1-intro.pdf (p.3)
TOP 4 [Score: 0.1306] - Doc: 5-nosql.pdf (p.34) 

Conclusion:
L'expérience montre que Word2Vec produit des scores de similarité cosinus très bas (~0.18) comparé à SBERT (~0.60).
Cela s'explique par l'approche 'Bag-of-Words' (moyenne des vecteurs) de Word2Vec qui dilue l'information avec les mots de 
liaison de la requête, et par la taille restreinte de notre corpus d'entraînement local par rapport au modèle SBERT 
pré-entraîné.


SBERT produit des scores hauts (~0.60) car il crée des vecteurs denses (384 dimensions) qui capturent le sens global, même
si les mots exacts diffèrent.
TF-IDF produit des scores bas (~0.17) car il génère des vecteurs creux (sparses) extrêmement grands où seuls les mots-clés
exacts créent une intersection mathématique.

"""

'\nQuestion = "What is a Neural Network?"\n\nscore Sbert\nTOP 1 [Score: 0.6328] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)\nTOP 2 [Score: 0.5829] - Doc: A Neural Network Playground.pdf (p.1)\nTOP 3 [Score: 0.5711] - Doc: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf (p.117)\nTOP 4 [Score: 0.5706] - Doc: Intro_DL_H515_2026.pdf (p.25)\n\n\nscore word2vec\nTOP 1 [Score: 0.1938] - Doc: 1-intro.pdf (p.2)\nTOP 2 [Score: 0.1741] - Doc: intro.pdf (p.3)\nTOP 3 [Score: 0.1694] - Doc: 1-intro.pdf (p.19)\nTOP 4 [Score: 0.1607] - Doc: 1-intro.pdf (p.30)\n\n\nscore tfidf\nTOP 1 [Score: 0.1719] - Doc: Intro_DL_H515_2026.pdf (p.6)\nTOP 2 [Score: 0.1466] - Doc: A Neural Network Playground.pdf (p.1)\nTOP 3 [Score: 0.1329] - Doc: 1-intro.pdf (p.3)\nTOP 4 [Score: 0.1306] - Doc: 5-nosql.pdf (p.34) \n\nConclusion:\n\n\n'

In [5]:
# ==============================================================================
# CELLULE 3 : GÉNÉRATION DES QUESTIONS D'EXAMEN (LLM LOCAL)
# ==============================================================================

def generate_exam_with_llm(query: str, spark_results, num_questions: int = 3):
    """
    Construit un prompt sourcé et interroge Ollama.
    """
    if not spark_results:
        return "Aucun document n'a été trouvé pour générer l'examen."

    # 1. Construction du contexte avec les métadonnées (Traçabilité exigée)
    context_text = ""
    for idx, row in enumerate(spark_results, 1):
        context_text += f"\n[{idx}] (Source_pdf: {row['source_pdf']}, Page {row['page_num']}, ID: {row['chunk_id']})\n{row['chunk_text']}\n"
    
    # 2. Ingénierie du Prompt (Prompt strict pour limiter les hallucinations)
    prompt = f"""You are a strict university teaching assistant designing an exam.
TASK: Generate {num_questions} exam questions about the topic: "{query}".

RULES:
- Base your questions and answers EXCLUSIVELY on the provided SOURCES.
- If the sources do not contain enough information, state: "Insufficient information in the provided sources."
- You MUST append citations at the end of each question and each answer.

OUTPUT FORMAT:
Q1. [Question text here] [Source_pdf,Page,ID]
A1. [Answer text here] [Source_pdf,Page,ID]

SOURCES:
{context_text}
"""
    
    # EXIGENCE DU PROJET : Afficher les paramètres et le prompt exact
    print("\n" + "-"*70)
    print("⚙️ PARAMÈTRES DE GÉNÉRATION UTILISÉS :")
    print(f"- Backend : {generator_backend}")
    print(f"- Modèle : {generator_model}")
    print(f"- Max context tokens : {max_context_tokens}")
    print("-" * 70)
    print("📝 PROMPT EXACT ENVOYÉ À L'IA :")
    print(prompt)
    print("-" * 70)
    # 3. Appel de l'API Ollama
    print("\n⏳ Envoi du prompt à Ollama (modèle local)...")
    try:
        response = requests.post(
            "http://host.docker.internal:11434/api/generate",
            json={
                "model": "llama3.2:3b",
                "prompt": prompt,
                "stream": False
            },
            timeout=120
        )
        response.raise_for_status()
        return response.json()["response"]
    
    except requests.exceptions.RequestException as e:
        return f"Erreur de connexion à l'API Ollama.{e}"

# --- GÉNÉRATION FINALE ---
examen_genere = generate_exam_with_llm(
    query=query, 
    spark_results=resultats, 
    num_questions=number_of_questions_to_generate
)

print("\n" + "="*70)
print("🎓 EXAMEN GÉNÉRÉ PAR L'IA :")
print("="*70)
print(examen_genere)


----------------------------------------------------------------------
⚙️ PARAMÈTRES DE GÉNÉRATION UTILISÉS :
- Backend : ollama
- Modèle : llama3.2:3b
- Max context tokens : 2048
----------------------------------------------------------------------
📝 PROMPT EXACT ENVOYÉ À L'IA :
You are a strict university teaching assistant designing an exam.
TASK: Generate 2 exam questions about the topic: "What is a Neural Network?".

RULES:
- Base your questions and answers EXCLUSIVELY on the provided SOURCES.
- If the sources do not contain enough information, state: "Insufficient information in the provided sources."
- You MUST append citations at the end of each question and each answer.

OUTPUT FORMAT:
Q1. [Question text here] [Source_pdf,Page,ID]
A1. [Answer text here] [Source_pdf,Page,ID]

SOURCES:

[1] (Source_pdf: a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf, Page 117, ID: p117_c1)
activation function of the hidden layer neurons vector Z = weights between the hi